# Google Colab: Qwen + AnyServe

This notebook keeps the whole example inside one Colab runtime:

1. load `Qwen/Qwen3-0.6B` from Hugging Face with `transformers`
2. expose it as a tiny OpenAI-compatible upstream on `127.0.0.1:8000`
3. start `anyserve` on the same notebook
4. optionally publish the AnyServe gateway with Cloudflare Tunnel so another person can call it

Notes:

- this notebook is chat-only
- it uses `Qwen/Qwen3-0.6B`, which matches the checked-in Ollama example's Qwen3 family
- `stream=true` is not implemented in the Hugging Face shim
- pick a small public model that fits the GPU the notebook runtime actually gives you
- if Colab did not assign a GPU, generation still works but will be much slower


In [ ]:
%pip install -q fastapi uvicorn transformers accelerate sentencepiece requests
!curl -fsSL https://raw.githubusercontent.com/anyserve/anyserve/main/scripts/install.sh | sh -s -- --dir /usr/local/bin
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared


In [ ]:
import json
import os
import pathlib
import textwrap

MODEL_ID = os.environ.get("HF_MODEL_ID", "Qwen/Qwen3-0.6B")
UPSTREAM_PORT = 8000
GATEWAY_PORT = 8080
GRPC_PORT = 50052

WORKDIR = pathlib.Path("/content/anyserve-colab")
WORKDIR.mkdir(parents=True, exist_ok=True)

(WORKDIR / "anyserve.toml").write_text(
    textwrap.dedent(
        f"""
        [server]
        grpc_host = "0.0.0.0"
        grpc_port = {GRPC_PORT}
        heartbeat_ttl_secs = 30
        default_lease_ttl_secs = 30

        [openai]
        enabled = true
        listen = "0.0.0.0:{GATEWAY_PORT}"
        models = [{json.dumps(MODEL_ID)}]
        chat_interface = "llm.chat.v1"
        request_timeout_secs = 120

        [openai.demand.required_attributes]
        family = "llm"
        protocol = "openai-compatible"
        """
    ).strip()
    + "\n"
)

(WORKDIR / "worker.toml").write_text(
    textwrap.dedent(
        f"""
        base_url = "http://127.0.0.1:{UPSTREAM_PORT}/v1"
        provider = "huggingface-transformers"
        interfaces = ["llm.chat.v1"]
        poll_interval_secs = 1
        heartbeat_interval_secs = 5
        upstream_timeout_secs = 120

        [attributes]
        region = "colab"

        [capacity]
        slot = 1
        """
    ).strip()
    + "\n"
)

hf_server = '''
import os
import time
import uuid
from typing import Any, Dict, List

import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.environ.get("HF_MODEL_ID", "Qwen/Qwen3-0.6B")
PORT = int(os.environ.get("PORT", "8000"))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

app = FastAPI()


class ChatCompletionRequest(BaseModel):
    model: str | None = None
    messages: List[Dict[str, Any]]
    max_tokens: int | None = 256
    temperature: float | None = 0.7
    top_p: float | None = 0.95
    stream: bool | None = False


def render_prompt(messages: List[Dict[str, Any]]) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            try:
                return tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=False,
                )
            except TypeError:
                return tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
        except Exception:
            pass

    lines = []
    for message in messages:
        role = message.get("role", "user")
        content = message.get("content", "")
        if isinstance(content, list):
            chunks = []
            for item in content:
                if isinstance(item, dict) and item.get("type") == "text":
                    chunks.append(item.get("text", ""))
            content = "\n".join(chunks)
        lines.append(f"{role}: {content}")
    lines.append("assistant:")
    return "\n".join(lines)


@app.get("/healthz")
def healthz() -> Dict[str, Any]:
    return {"ok": True, "model": MODEL_ID}


@app.get("/v1/models")
def list_models() -> Dict[str, Any]:
    created = int(time.time())
    return {
        "object": "list",
        "data": [
            {
                "id": MODEL_ID,
                "object": "model",
                "created": created,
                "owned_by": "huggingface",
            }
        ],
    }


@app.post("/v1/chat/completions")
def chat(request: ChatCompletionRequest) -> Dict[str, Any]:
    if request.stream:
        raise HTTPException(
            status_code=400,
            detail="stream=true is not implemented in this notebook example",
        )

    if request.model and request.model != MODEL_ID:
        raise HTTPException(
            status_code=400,
            detail=f"this notebook only serves {MODEL_ID}",
        )

    prompt = render_prompt(request.messages)
    batch = tokenizer(prompt, return_tensors="pt")
    batch = {name: tensor.to(model.device) for name, tensor in batch.items()}

    do_sample = (request.temperature or 0.0) > 0
    max_tokens = request.max_tokens or 256

    generate_kwargs = {
        **batch,
        "max_new_tokens": max_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        generate_kwargs["temperature"] = request.temperature if request.temperature is not None else 0.7
        generate_kwargs["top_p"] = request.top_p if request.top_p is not None else 0.95

    with torch.inference_mode():
        output_ids = model.generate(**generate_kwargs)

    new_tokens = output_ids[0, batch["input_ids"].shape[1] :]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    created = int(time.time())

    return {
        "id": f"chatcmpl-{uuid.uuid4().hex}",
        "object": "chat.completion",
        "created": created,
        "model": MODEL_ID,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "stop",
            }
        ],
        "usage": {
            "prompt_tokens": int(batch["input_ids"].shape[1]),
            "completion_tokens": int(new_tokens.shape[0]),
            "total_tokens": int(batch["input_ids"].shape[1] + new_tokens.shape[0]),
        },
    }


if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")
'''

(WORKDIR / "hf_openai_server.py").write_text(textwrap.dedent(hf_server).strip() + "\n")

print(f"workdir: {WORKDIR}")
print((WORKDIR / "anyserve.toml").read_text())
print((WORKDIR / "worker.toml").read_text())


In [ ]:
import os
import subprocess
import time

import requests

processes = {}


def start_process(name, cmd, extra_env=None):
    log_path = WORKDIR / f"{name}.log"
    proc = subprocess.Popen(
        cmd,
        cwd=WORKDIR,
        env={**os.environ, **(extra_env or {})},
        stdout=log_path.open("w"),
        stderr=subprocess.STDOUT,
        start_new_session=True,
        text=True,
    )
    processes[name] = {"proc": proc, "log": log_path}
    print(f"started {name}: pid={proc.pid}, log={log_path}")
    return proc


def wait_http(url, timeout=180):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(url, timeout=5)
            if response.ok:
                return response
        except Exception as exc:
            last_error = exc
        time.sleep(2)
    raise RuntimeError(f"timed out waiting for {url}: {last_error}")


def tail(name, lines=40):
    log_path = processes[name]["log"]
    text = log_path.read_text() if log_path.exists() else ""
    print("\n".join(text.splitlines()[-lines:]))


start_process(
    "hf-server",
    ["python", "hf_openai_server.py"],
    {"HF_MODEL_ID": MODEL_ID, "PORT": str(UPSTREAM_PORT)},
)
wait_http(f"http://127.0.0.1:{UPSTREAM_PORT}/healthz", timeout=900)
requests.get(f"http://127.0.0.1:{UPSTREAM_PORT}/v1/models", timeout=30).json()


In [ ]:
start_process("anyserve-serve", ["anyserve", "serve", "--config", "anyserve.toml"])
wait_http(f"http://127.0.0.1:{GATEWAY_PORT}/healthz", timeout=120)

start_process(
    "anyserve-worker",
    [
        "anyserve",
        "--endpoint",
        f"http://127.0.0.1:{GRPC_PORT}",
        "worker",
        "--config",
        "worker.toml",
    ],
)
wait_http(f"http://127.0.0.1:{GATEWAY_PORT}/readyz", timeout=120)
print("AnyServe is ready.")


In [ ]:
local_base_url = f"http://127.0.0.1:{GATEWAY_PORT}"

print(requests.get(f"{local_base_url}/v1/models", timeout=30).json())

response = requests.post(
    f"{local_base_url}/v1/chat/completions",
    json={
        "model": MODEL_ID,
        "messages": [
            {
                "role": "user",
                "content": "Reply with one short sentence saying this notebook is fronted by AnyServe.",
            }
        ],
    },
    timeout=300,
)
response.raise_for_status()
response.json()


In [ ]:
import re

start_process(
    "cloudflared",
    [
        "cloudflared",
        "tunnel",
        "--url",
        f"http://127.0.0.1:{GATEWAY_PORT}",
        "--no-autoupdate",
    ],
)

pattern = re.compile(r"https://[a-z0-9-]+\\.trycloudflare\\.com")
deadline = time.time() + 90
public_base_url = None

while time.time() < deadline and public_base_url is None:
    log_text = processes["cloudflared"]["log"].read_text() if processes["cloudflared"]["log"].exists() else ""
    match = pattern.search(log_text)
    if match:
        public_base_url = match.group(0)
        break
    time.sleep(2)

if public_base_url is None:
    tail("cloudflared")
    raise RuntimeError("could not find a public trycloudflare URL")

print(public_base_url)


In [ ]:
snippet = textwrap.dedent(
    f"""
    Public base URL:
    {public_base_url}

    Another person can call the gateway with:

    curl {public_base_url}/v1/models

    curl {public_base_url}/v1/chat/completions \\
      -H 'content-type: application/json' \\
      -d '{json.dumps({"model": MODEL_ID, "messages": [{"role": "user", "content": "hello from another laptop"}]})}'
    """
).strip()

print(snippet)


In [ ]:
import signal

for name, info in processes.items():
    proc = info["proc"]
    if proc.poll() is None:
        os.killpg(proc.pid, signal.SIGTERM)
        print(f"stopped {name}")
